# Transformer HF/TMF Simulation Suite -- Colab launcher

Runs the project's NGSolve simulation stages (CPU and GPU, DC and AC) by cloning [github.com/seifhamzaoui/hf-simulation](https://github.com/seifhamzaoui/hf-simulation) -- one cell per stage, no desktop GUI needed. Result files (the generated `.mat` matrices) are saved to a Google Drive folder at the end.

**Not included here:** `transformer_geometry.py` / `transformer_geometry_rectangular.py` (the geometry BUILDER scripts). They open Netgen's own 3D viewer window (`from netgen.gui import *` + `Draw(...)`), which needs a real display -- Colab is headless and has none. Build/regenerate geometry locally (or via `simulation_ui.py`'s Geometry Builder screen), commit the resulting `.step` files, and push; this notebook only ever *reads* them.

**GPU cells** need a GPU runtime: `Runtime > Change runtime type > T4 GPU` (or better) before running them.

**Heads up:** `run_capacitance()` and `run_dc_resistance()` (CPU) have a built-in `input("...press Enter to continue")` safety pause before their expensive step -- Colab shows this as a small inline text box under the cell; type anything and hit Enter (or just Enter) to continue. Every other function here runs straight through with no prompt.

## 1. Clone the project from GitHub
If the repo is **private**, plain `git clone` will fail with an authentication error -- generate a Personal Access Token (GitHub Settings > Developer settings > Personal access tokens, `repo` scope) and clone with `https://<TOKEN>@github.com/seifhamzaoui/hf-simulation.git` instead. Paste the token directly into this cell for your own session only; don't commit it anywhere.

In [1]:
import os
import sys

REPO_URL = "https://github.com/seifhamzaoui/hf-simulation.git"
PROJECT_DIR = "/content/hf-simulation"

!git clone {REPO_URL} "{PROJECT_DIR}"

assert os.path.isdir(PROJECT_DIR), f"Clone failed -- check REPO_URL above (private repo needs a token, see markdown above)"
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print("cwd:", os.getcwd())
print("config.py found:", os.path.isfile("config.py"))
print("transformer_model_closed.step found:", os.path.isfile("transformer_model_closed.step"))

fatal: destination path '/content/hf-simulation' already exists and is not an empty directory.
cwd: /content/hf-simulation
config.py found: True
transformer_model_closed.step found: True


## 2. Mount Google Drive (where result files will be saved)
Results are copied here at the end -- nothing is written back to the GitHub repo.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os

RESULTS_DRIVE_DIR = "/content/drive/MyDrive/ngsolve_results"  # <-- edit if you want a different folder
os.makedirs(RESULTS_DRIVE_DIR, exist_ok=True)
print("Results will be saved to:", RESULTS_DRIVE_DIR)

Results will be saved to: /content/drive/MyDrive/ngsolve_results


## 3. Install dependencies
NGSolve/Netgen ship proper Linux wheels, so this is a plain `pip install` on Colab (no portable-runtime tricks needed, unlike the Windows .exe packaging).

In [4]:
!pip install -q ngsolve ezdxf

## 4. CPU Stages -- `simulation_ngsolve.py`
Capacitance (Electrostatics), DC Resistance (DC Conduction), Inductance (curl-curl field solve).

In [5]:
import simulation_ngsolve as sim

In [ ]:
# Capacitance (Electrostatics) -- saves cap_data.mat
sim.run_capacitance()

In [ ]:
# DC Resistance (DC Conduction) -- saves DCR.mat
sim.run_dc_resistance()

In [ ]:
# Inductance (curl-curl field solve, closed rings) -- saves induc.mat
# Pass test_rings=["ringp1"] for a fast single-ring smoke test instead of the full N-ring matrix.
sim.run_inductance()

*** Mesh-converged but still ~2.1-2.3x Q3D's absolute scale for a lone
*** ring against the real core -- see this function's docstring before
*** trusting absolute values; coupling structure should be more reliable.
Loading closed-ring geometry...
Refined 4 entrefer-facing core face(s) to maxh=2.0000mm


## 5. GPU Stages -- `simulation_ngsolve_cuda.py`
**Needs a GPU runtime** (`Runtime > Change runtime type > T4 GPU`) before running this section.

In [6]:
!pip install -q cupy-cuda12x
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-90ceb529-5026-b2e2-bd4d-c3614aa5a268)


In [7]:
import simulation_ngsolve_cuda as simgpu

In [ ]:
# Capacitance -- GPU solve
simgpu.run_capacitance_gpu()

In [ ]:
# DC Resistance -- GPU solve
simgpu.run_dc_resistance_gpu()

In [8]:
# Inductance (curl-curl field solve, closed rings) -- GPU solve
# Pass test_rings=["ringp1"] for a fast single-ring smoke test instead of the full N-ring matrix.
simgpu.run_inductance_gpu()

Loading closed-ring geometry...
Refined 4 entrefer-facing core face(s) to maxh=2.0000mm
Meshing (whole assembly, ~2 minutes)...
mesh: 874924 elements
Building each ring's current source (independent EMF-driven conduction solves)...
  1/60 ringp1: current source ready
  2/60 ringp2: current source ready
  3/60 ringp3: current source ready
  4/60 ringp4: current source ready
  5/60 ringp5: current source ready
  6/60 ringp6: current source ready
  7/60 ringp7: current source ready
  8/60 ringp8: current source ready
  9/60 ringp9: current source ready
  10/60 ringp10: current source ready
  11/60 ringp11: current source ready
  12/60 ringp12: current source ready
  13/60 ringp13: current source ready
  14/60 ringp14: current source ready
  15/60 ringp15: current source ready
  16/60 ringp16: current source ready
  17/60 ringp17: current source ready
  18/60 rings1: current source ready
  19/60 rings2: current source ready
  20/60 rings3: current source ready
  21/60 rings4: current sourc

KeyError: 'ac'

In [ ]:
# PEEC self-inductance cross-checks (ringp1 only, free space / with core BEM correction) -- GPU
simgpu.run_inductance_peec_gpu("ringp1")
simgpu.run_inductance_peec_core_gpu("ringp1")

## 6. AC Litz Sweep (CPU) -- `simulation_ngsolve_litz.py`
Full N-ring x frequency-sweep AC inductance/resistance -- far more expensive than the quick single-ring test. Start with the smoke test.

In [ ]:
import simulation_ngsolve_litz as lz

# Quick smoke test (single ring). Call lz.run_litz_sweep() with no args for the FULL sweep
# (all rings x config.sim_frequencies) -- see this function's own docstring for the cost warning first.
lz.run_litz_sweep(test_rings=["ringp1"])

## 7. R/L Ratio Sweep (CPU) -- `simulation_ngsolve_litz_ratio.py`
Cheaper AC/DC ratio sweep from a small representative turn sample, used to scale the full DCR.mat/induc.mat matrices.

In [ ]:
import simulation_ngsolve_litz_ratio as lrg

# primary_count/secondary_count default to config.py's LITZ_RATIO_SAMPLE_COUNT_PRIMARY/_SECONDARY.
lrg.run_ratio_sweep()

## 8. R/L Ratio Sweep (GPU) -- `simulation_ngsolve_litz_ratio_cuda.py`
**Known issue** (from this project's own dev history): the AC (f>0) GPU solve currently fails to converge/factor (ILU+GMRES never got a working preconditioner) -- only the DC (f=0) baseline is confirmed working. Expect a real sweep to error out partway through. Needs the GPU runtime + `cupy` from section 5.

In [ ]:
import simulation_ngsolve_litz_ratio_cuda as lrgpu

lrgpu.run_ratio_sweep_gpu()

## 9. Save result files to Google Drive
Every stage above saves its output `.mat` file into `PROJECT_DIR/ngsolve matrices/` (all of them share that one folder -- `sim.MATRIX_DIR`). This copies just that folder to `RESULTS_DRIVE_DIR` from section 2.

In [ ]:
import shutil
import os

src = os.path.join(PROJECT_DIR, "ngsolve matrices")
shutil.copytree(src, RESULTS_DRIVE_DIR, dirs_exist_ok=True)
print(f"Copied {src} -> {RESULTS_DRIVE_DIR}")
print(os.listdir(RESULTS_DRIVE_DIR))